# Sprint 2 — Modeling, Interpretation & Responsible AI

**Purpose:** Train a leakage-safe XGBoost churn classifier on `training_data.csv`, beat a naive baseline, log MLflow runs, and produce SHAP / fairness / drift evidence for Sam and Jamie.

**Depends on:** Sprint 1 export `Data/processed/training_data.csv` (ADR 004 label; month-`t` features only).

**Anti-patterns (must obey):**
- Split **only** on `Month` (chronological). Never `train_test_split(..., shuffle=True)`.
- Do **not** StandardScaler balances (±1e10 outliers); trees tolerate raw scale + native NaNs (`Active_Txn_Lag3`).
- Accuracy alone is not success (~88% always-negative baseline).

**Inputs:** `Data/processed/training_data.csv`  
**Outputs:** MLflow SQLite `mlflow.db` (+ Yellowbrick/SHAP artifacts in `mlartifacts/`), Evidently at `Docs/Sprint2/data_drift_report.html`, Fairlearn CSV under `Docs/Sprint2/reports/`, metrics JSON for the model card.

**Column names:** snake_case — see `Docs/Data/Data_Dictionary.md` §2.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import shap
import xgboost as xgb
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8-whitegrid")

# Notebook lives in Code/Sprint2/ — resolve paths from repo root regardless of kernel cwd.
REPO_ROOT = Path("../..").resolve()
DATA_PATH = REPO_ROOT / "Data" / "processed" / "training_data.csv"
REPORT_DIR = REPO_ROOT / "Docs" / "Sprint2" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
# MLflow ≥3 blocks the legacy ./mlruns file store unless MLFLOW_ALLOW_FILE_STORE=true.
# Use SQLite tracking (recommended) — artifacts still land under mlartifacts/.
MLFLOW_DB = REPO_ROOT / "mlflow.db"
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB.resolve().as_posix()}"

assert DATA_PATH.exists(), (
    f"Missing {DATA_PATH}. Re-run Code/Sprint1/02_feature_engineering.ipynb first."
)

TARGET = "Is_Churned"
ID_COLS = ["customer_id", "Month"]  # kept for splits / join-back; not model features
CAT_COLS = ["state"]
# Primary ranking metric for "beats baseline" (imbalance-aware).
PRIMARY_METRIC = "pr_auc"

RANDOM_STATE = 42
# SHAP / Fairlearn on full 2.2M rows is too slow for interactive Run All — sample.
SHAP_SAMPLE_N = 20_000
FAIRNESS_SAMPLE_N = 100_000

print(f"REPO_ROOT={REPO_ROOT}")
print(f"DATA_PATH={DATA_PATH}")

## 1. Load training frame & schema check

Reload the Sprint 1 export. Eligible rows only were written (tenure ≥ 2, next month known). Confirm target non-null and positive rate matches the FE lock (~11.76%).

In [ ]:
df = pd.read_csv(DATA_PATH)
df["Month"] = pd.PeriodIndex(df["Month"], freq="M")

print(f"rows={len(df):,}  cols={df.shape[1]}")
print(f"months={df['Month'].min()} → {df['Month'].max()}  (n={df['Month'].nunique()})")
print(f"customers={df['customer_id'].nunique():,}")
print(f"pos_rate={df[TARGET].mean():.4%}")
print(f"Active_Txn_Lag3 nulls={df['Active_Txn_Lag3'].isna().sum():,} (expected early tenure)")
assert df[TARGET].notna().all(), "Target has nulls — FE export should be eligible-only"
assert abs(df[TARGET].mean() - 0.117601) < 1e-4, "Positive rate drifted from Sprint 1 lock"
df.head(3)

### Analysis — load

Expect ~2.26M rows × 27 cols, months `2007-03`→`2020-04`, pos rate **11.7601%**.  
`Active_Txn_Lag3` NaNs are intentional (grace kills Lag1/Lag2 nulls only). Leave them for XGBoost native missing handling — do not drop rows.

## 2. Temporal train / test split (no shuffle)

**Rule:** every test month is strictly after every train month.  
We take the earliest **80% of unique months** for train and the latest **20%** for test (~32 months holdout). Validation for light threshold tuning is the last 10% of *train* months (still before test).

This matches `Docs/Sprint1/ML_Anti_Patterns.md` — random row splits would leak future months into train.

In [ ]:
months_sorted = sorted(df["Month"].unique())
n_months = len(months_sorted)
n_train_months = int(np.floor(0.80 * n_months))
n_val_months = max(1, int(np.floor(0.10 * n_train_months)))

train_months_all = months_sorted[:n_train_months]
val_months = train_months_all[-n_val_months:]
fit_months = train_months_all[:-n_val_months]
test_months = months_sorted[n_train_months:]

fit_df = df[df["Month"].isin(fit_months)].copy()
val_df = df[df["Month"].isin(val_months)].copy()
test_df = df[df["Month"].isin(test_months)].copy()

print("=== Temporal split ===")
print(f"fit : {fit_months[0]} → {fit_months[-1]}  rows={len(fit_df):,}  pos={fit_df[TARGET].mean():.4%}")
print(f"val : {val_months[0]} → {val_months[-1]}  rows={len(val_df):,}  pos={val_df[TARGET].mean():.4%}")
print(f"test: {test_months[0]} → {test_months[-1]}  rows={len(test_df):,}  pos={test_df[TARGET].mean():.4%}")

# Hard guards against accidental future leakage into fit.
assert max(fit_months) < min(val_months) <= max(val_months) < min(test_months)
assert fit_df["Month"].max() < val_df["Month"].min()
assert val_df["Month"].max() < test_df["Month"].min()
print("Temporal ordering OK.")

### Analysis — split

Test prevalence may differ from train (concept drift over ~13 years). That is expected and is why we later run Evidently. If test pos-rate collapses or spikes, note it before trusting a single AUC number.

## 3. Feature matrix

- Drop `customer_id` / `Month` from `X` (identity / split keys only).
- One-hot `state` **fit on train only** (no test leakage into categories).
- Keep `gone_this_month` — known at end of month `t`; label is still `t+1`.
- No scaling.

In [ ]:
feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET]]
num_cols = [c for c in feature_cols if c not in CAT_COLS]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("num", "passthrough", num_cols),
    ]
)

X_fit = fit_df[feature_cols]
y_fit = fit_df[TARGET].astype(int)
X_val = val_df[feature_cols]
y_val = val_df[TARGET].astype(int)
X_test = test_df[feature_cols]
y_test = test_df[TARGET].astype(int)

preprocess.fit(X_fit)
X_fit_m = preprocess.transform(X_fit)
X_val_m = preprocess.transform(X_val)
X_test_m = preprocess.transform(X_test)

ohe = preprocess.named_transformers_["cat"]
ohe_names = list(ohe.get_feature_names_out(CAT_COLS))
model_feature_names = ohe_names + num_cols
print(f"raw features={len(feature_cols)}  after OHE={len(model_feature_names)}")
print("numeric (passthrough):", num_cols[:8], "...")

## 4. Naive baselines

1. **Always-negative** — predict 0 for everyone (accuracy ≈ 1 − prevalence).
2. **Impending-exit heuristic** — predict 1 iff `gone_this_month == 1` (already quiet-at-zero at `t`).

XGBoost must beat these on **PR-AUC** (primary) and **ROC-AUC** (secondary). High accuracy alone is not a win.

In [ ]:
def metrics_dict(y_true, y_prob, y_pred, label: str) -> dict:
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = np.asarray(y_pred)
    # Constant scores (always-negative) make ROC-AUC undefined — return nan, not a crash.
    if np.unique(y_prob).size < 2:
        roc = float("nan")
    else:
        roc = float(roc_auc_score(y_true, y_prob))
    out = {
        "label": label,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc,
        "pr_auc": float(average_precision_score(y_true, y_prob)),
    }
    return out


def print_metrics(m: dict) -> None:
    print(
        f"{m['label']:28s}  "
        f"acc={m['accuracy']:.4f}  prec={m['precision']:.4f}  "
        f"rec={m['recall']:.4f}  f1={m['f1']:.4f}  "
        f"roc_auc={m['roc_auc']:.4f}  pr_auc={m['pr_auc']:.4f}"
    )


# Probabilities for ranking metrics: constant 0, or the binary heuristic itself.
y_prob_neg = np.zeros(len(y_test), dtype=float)
y_pred_neg = np.zeros(len(y_test), dtype=int)
base_neg = metrics_dict(y_test, y_prob_neg, y_pred_neg, "always_negative")

y_prob_heur = test_df["gone_this_month"].astype(float).to_numpy()
y_pred_heur = (y_prob_heur >= 0.5).astype(int)
base_heur = metrics_dict(y_test, y_prob_heur, y_pred_heur, "gone_this_month_heuristic")

print("=== Test baselines ===")
print_metrics(base_neg)
print_metrics(base_heur)
print(f"\nPrimary metric for beat-baseline: {PRIMARY_METRIC}")

### Analysis — baselines

Always-negative will show high accuracy and **undefined/zero** ranking quality (constant scores → treat PR/ROC carefully; sklearn still returns prevalence-based AP for constant 0).  
The `gone_this_month` heuristic is the real bar: it uses a strong current-month signal without peeking at `t+1`. XGBoost should improve PR-AUC by combining balance trajectory + dormancy lags.

## 5. XGBoost + MLflow

- `scale_pos_weight = n_neg / n_pos` on the fit set (class imbalance ~11.8%).
- Native NaN support — no imputer for Lag3.
- Light early stopping on the temporal val window.
- Log params, metrics, and the booster to a local SQLite store (`mlflow.db`).
- Yellowbrick / SHAP figures are logged onto **this same run** in later cells via `run_id`.


In [ ]:
n_pos = int(y_fit.sum())
n_neg = int(len(y_fit) - n_pos)
scale_pos_weight = n_neg / max(n_pos, 1)

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": ["aucpr", "auc"],
    "max_depth": 6,
    "learning_rate": 0.08,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 50,
    "reg_lambda": 1.0,
    "scale_pos_weight": float(round(scale_pos_weight, 4)),
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

dfit = xgb.DMatrix(X_fit_m, label=y_fit, feature_names=model_feature_names)
dval = xgb.DMatrix(X_val_m, label=y_val, feature_names=model_feature_names)
dtest = xgb.DMatrix(X_test_m, label=y_test, feature_names=model_feature_names)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow tracking URI={MLFLOW_TRACKING_URI}")
mlflow.set_experiment("meridian_churn_sprint2")

with mlflow.start_run(run_name="xgb_temporal_v2_ops_threshold") as run:
    # Log once only — MLflow forbids changing a param key on the same run.
    # eval_metric is a list for XGBoost; stringify for the tracking store.
    loggable = {
        **{k: (",".join(v) if isinstance(v, list) else v) for k, v in xgb_params.items()},
        "n_estimators_max": 400,
        "early_stopping_rounds": 40,
        "fit_months": f"{fit_months[0]}:{fit_months[-1]}",
        "val_months": f"{val_months[0]}:{val_months[-1]}",
        "test_months": f"{test_months[0]}:{test_months[-1]}",
        "n_fit": len(y_fit),
        "n_test": len(y_test),
    }
    mlflow.log_params(loggable)

    booster = xgb.train(
        params=xgb_params,
        dtrain=dfit,
        num_boost_round=400,
        evals=[(dfit, "fit"), (dval, "val")],
        early_stopping_rounds=40,
        verbose_eval=50,
    )

    # Threshold: ops policy on validation only (never tune on test).
    # Locked rule — Docs/Sprint2/ML_Learnings.md §2.4:
    #   precision >= 0.40 AND selection_rate <= capacity_cap, maximize recall.
    # First-run max-F1 (t=0.475) was recall-greedy (~21% precision) — not ops-ready.
    val_prob = booster.predict(dval)
    PREC_FLOOR = 0.40
    CAPACITY_CAPS = (0.05, 0.02, 0.01)
    thresholds = np.linspace(0.05, 0.99, 95)
    grid_rows = []
    for t in thresholds:
        pred_t = (val_prob >= t).astype(int)
        grid_rows.append(
            {
                "t": float(t),
                "sel": float(pred_t.mean()),
                "prec": float(precision_score(y_val, pred_t, zero_division=0)),
                "rec": float(recall_score(y_val, pred_t, zero_division=0)),
                "f1": float(f1_score(y_val, pred_t, zero_division=0)),
            }
        )
    grid = pd.DataFrame(grid_rows)
    chosen = None
    threshold_rule = None
    for cap in CAPACITY_CAPS:
        feasible = grid[(grid["prec"] >= PREC_FLOOR) & (grid["sel"] <= cap)]
        if len(feasible):
            feasible = feasible.sort_values(
                ["rec", "prec", "f1"], ascending=[False, False, False]
            )
            chosen = feasible.iloc[0]
            threshold_rule = f"prec>={PREC_FLOOR} & sel<={cap} max_recall"
            break
    if chosen is None:
        # Fallback: top 5% by score on validation (capacity-first).
        best_t = float(np.quantile(val_prob, 0.95))
        pred_fb = (val_prob >= best_t).astype(int)
        threshold_rule = "fallback_quantile_top5pct"
        print("WARNING: no grid cut met precision+capacity; using top-5% quantile.")
        print(
            f"Chosen decision threshold (val {threshold_rule}): {best_t:.3f}  "
            f"sel={pred_fb.mean():.4f} prec={precision_score(y_val, pred_fb, zero_division=0):.4f}"
        )
    else:
        best_t = float(chosen["t"])
        print(
            f"Chosen decision threshold (val {threshold_rule}): {best_t:.3f}  "
            f"sel={chosen['sel']:.4f} prec={chosen['prec']:.4f} rec={chosen['rec']:.4f}"
        )

    # Deck diagnostic only — not used to pick t*.
    t_at5 = float(np.quantile(val_prob, 0.95))
    pred5 = (val_prob >= t_at5).astype(int)
    print(
        f"Val precision@5%={precision_score(y_val, pred5, zero_division=0):.4f}  "
        f"recall@5%={recall_score(y_val, pred5, zero_division=0):.4f}"
    )

    test_prob = booster.predict(dtest)
    test_pred = (test_prob >= best_t).astype(int)
    xgb_test = metrics_dict(y_test, test_prob, test_pred, "xgboost")

    mlflow.log_params(
        {
            "threshold_rule": threshold_rule,
            "precision_floor": PREC_FLOOR,
            "capacity_caps": ",".join(str(c) for c in CAPACITY_CAPS),
        }
    )
    mlflow.log_metrics(
        {
            "threshold": best_t,
            "val_precision": float(
                precision_score(y_val, (val_prob >= best_t).astype(int), zero_division=0)
            ),
            "val_recall": float(
                recall_score(y_val, (val_prob >= best_t).astype(int), zero_division=0)
            ),
            "val_selection_rate": float((val_prob >= best_t).mean()),
            "test_accuracy": xgb_test["accuracy"],
            "test_precision": xgb_test["precision"],
            "test_recall": xgb_test["recall"],
            "test_f1": xgb_test["f1"],
            "test_roc_auc": xgb_test["roc_auc"],
            "test_pr_auc": xgb_test["pr_auc"],
            "baseline_heur_pr_auc": base_heur["pr_auc"],
            "baseline_heur_roc_auc": base_heur["roc_auc"],
        }
    )
    mlflow.xgboost.log_model(booster, artifact_path="model")
    run_id = run.info.run_id

print("=== Test comparison ===")
print_metrics(base_neg)
print_metrics(base_heur)
print_metrics(xgb_test)

beat_pr = xgb_test["pr_auc"] > base_heur["pr_auc"]
beat_roc = xgb_test["roc_auc"] > base_heur["roc_auc"]
print(f"\nBeats heuristic on PR-AUC? {beat_pr}  ({xgb_test['pr_auc']:.4f} vs {base_heur['pr_auc']:.4f})")
print(f"Beats heuristic on ROC-AUC? {beat_roc} ({xgb_test['roc_auc']:.4f} vs {base_heur['roc_auc']:.4f})")
print(f"MLflow run_id={run_id}")
print(f"best_iteration={booster.best_iteration}")


### Analysis — XGBoost

Record whether PR-AUC / ROC-AUC clear the heuristic bar. If not, inspect threshold and class weight before adding features.  
Confusion matrix next: for retention spend, false positives cost campaign budget; false negatives cost AUM — Sam cares about both, Jamie about slice fairness.

In [ ]:
cm = confusion_matrix(y_test, test_pred)
print("Confusion matrix [ [TN FP] / [FN TP] ]:")
print(cm)
print()
print(classification_report(y_test, test_pred, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, test_prob, ax=axes[0], name="XGBoost")
axes[0].set_title("ROC (test)")
PrecisionRecallDisplay.from_predictions(y_test, test_prob, ax=axes[1], name="XGBoost")
axes[1].set_title("Precision-Recall (test)")
plt.tight_layout()
plt.show()

## 6. Yellowbrick visual diagnostics (+ MLflow artifacts)

Sklearn-compatible wrapper around the booster. Produce:
- `ClassificationReport`
- `ROCAUC`
- `PrecisionRecallCurve`
- `ConfusionMatrix`

Each figure is saved under `Docs/Sprint2/reports/` and logged to the active MLflow run as `yellowbrick/*`.


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from yellowbrick.classifier import (
    ClassificationReport,
    ConfusionMatrix,
    PrecisionRecallCurve,
    ROCAUC,
)


class XGBBoosterWrapper(BaseEstimator, ClassifierMixin):
    """Minimal sklearn façade so Yellowbrick can call predict / predict_proba."""

    _estimator_type = "classifier"

    def __init__(self, booster, threshold: float = 0.5, feature_names=None):
        self.booster = booster
        self.threshold = threshold
        self.feature_names = feature_names
        self.classes_ = np.array([0, 1])

    def fit(self, X, y=None):
        return self

    def get_params(self, deep=True):
        return {
            "booster": self.booster,
            "threshold": self.threshold,
            "feature_names": self.feature_names,
        }

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def predict_proba(self, X):
        d = xgb.DMatrix(X, feature_names=self.feature_names)
        p1 = self.booster.predict(d)
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


wrap = XGBBoosterWrapper(booster, threshold=best_t, feature_names=model_feature_names)
yb_paths = []

# Full test matrix is huge — Yellowbrick calls predict_proba repeatedly.
# Sample keeps plots representative without multi-minute hangs.
_yb_n = min(40_000, X_test_m.shape[0])
_yb_rng = np.random.default_rng(RANDOM_STATE)
_yb_idx = _yb_rng.choice(X_test_m.shape[0], size=_yb_n, replace=False)
X_yb = X_test_m[_yb_idx]
y_yb = y_test.to_numpy()[_yb_idx]
print(f"Yellowbrick evaluation sample n={_yb_n:,}")


def _yb_save(fig, name: str):
    path = REPORT_DIR / name
    fig.savefig(path, dpi=120, bbox_inches="tight")
    yb_paths.append(path)
    print(f"Wrote {path}")
    return path


def _run_yb(name, factory):
    """Isolate each visualizer so one Yellowbrick quirk does not kill the cell."""
    try:
        fig, ax = plt.subplots(figsize=(6, 4))
        viz = factory(ax)
        # fit sets target_type_/classes_; wrapper.fit is a no-op (model already trained).
        viz.fit(X_yb, y_yb)
        viz.score(X_yb, y_yb)
        viz.finalize()
        _yb_save(fig, name)
        plt.show()
    except Exception as exc:  # noqa: BLE001
        plt.close("all")
        print(f"Yellowbrick skipped {name}: {type(exc).__name__}: {exc}")


_run_yb(
    "yellowbrick_classification_report.png",
    lambda ax: ClassificationReport(
        wrap, classes=["stay", "churn"], support=True, ax=ax, force_model=True
    ),
)
_run_yb(
    "yellowbrick_rocauc.png",
    lambda ax: ROCAUC(
        wrap,
        classes=["stay", "churn"],
        ax=ax,
        force_model=True,
        binary=True,
        micro=False,
        macro=False,
    ),
)
_run_yb(
    "yellowbrick_precision_recall.png",
    lambda ax: PrecisionRecallCurve(
        wrap, classes=["stay", "churn"], ax=ax, force_model=True, micro=False
    ),
)
_run_yb(
    "yellowbrick_confusion_matrix.png",
    lambda ax: ConfusionMatrix(
        wrap, classes=["stay", "churn"], ax=ax, percent=False, force_model=True
    ),
)

if yb_paths:
    with mlflow.start_run(run_id=run_id):
        for p in yb_paths:
            mlflow.log_artifact(str(p), artifact_path="yellowbrick")
    print(f"Logged {len(yb_paths)} Yellowbrick figures to MLflow run {run_id}")
else:
    print("No Yellowbrick figures produced — continuing notebook.")


### Analysis — Yellowbrick

Use the report / PR curve to see whether precision or recall is weaker at `best_t`.  
Confusion matrix makes FP vs FN trade-offs explicit for retention spend. All four plots are also in MLflow under `yellowbrick/`.


## 7. SHAP — global + local explanations (sampled)

TreeExplainer on a random **test** sample:
1. Mean |SHAP| ranking + beeswarm summary (global drivers)
2. Waterfall plots for a high-probability churn and a stay case (local)
3. Figures logged to MLflow as `shap/*`


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_shap = min(SHAP_SAMPLE_N, X_test_m.shape[0])
idx = rng.choice(X_test_m.shape[0], size=n_shap, replace=False)
X_shap = X_test_m[idx]
y_shap = y_test.to_numpy()[idx]
prob_shap = test_prob[idx]

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_shap)
# Older SHAP returned [class0, class1] for binary trees; newer returns a single matrix.
if isinstance(shap_values, list):
    shap_values = shap_values[1]

base_value = explainer.expected_value
if isinstance(base_value, (list, tuple, np.ndarray)):
    base_arr = np.asarray(base_value).reshape(-1)
    base_value = float(base_arr[-1])
else:
    base_value = float(base_value)

mean_abs = np.abs(shap_values).mean(axis=0)
shap_rank = (
    pd.Series(mean_abs, index=model_feature_names)
    .sort_values(ascending=False)
    .head(15)
)
print("Top SHAP features (mean |value|):")
print(shap_rank.to_string())

shap_paths = []

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_shap, feature_names=model_feature_names, show=False, max_display=15
)
plt.title(f"SHAP summary (test sample n={n_shap:,})")
plt.tight_layout()
summary_path = REPORT_DIR / "shap_summary.png"
plt.savefig(summary_path, dpi=120, bbox_inches="tight")
shap_paths.append(summary_path)
plt.show()
print(f"Wrote {summary_path}")

i_churn = int(np.argmax(prob_shap))
i_stay = int(np.argmin(prob_shap))


def _waterfall(i: int, tag: str):
    explanation = shap.Explanation(
        values=np.asarray(shap_values[i]).astype(float),
        base_values=base_value,
        data=np.asarray(X_shap[i]).astype(float),
        feature_names=list(model_feature_names),
    )
    shap.plots.waterfall(explanation, max_display=12, show=False)
    path = REPORT_DIR / f"shap_waterfall_{tag}.png"
    # shap.plots.waterfall draws on the current figure; grab it explicitly.
    fig = plt.gcf()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    shap_paths.append(path)
    plt.show()
    print(
        f"Waterfall {tag}: row_in_sample={i}  y={y_shap[i]}  "
        f"p_hat={prob_shap[i]:.4f}  -> {path}"
    )


_waterfall(i_churn, "high_risk")
_waterfall(i_stay, "low_risk")

with mlflow.start_run(run_id=run_id):
    for p in shap_paths:
        mlflow.log_artifact(str(p), artifact_path="shap")
print(f"Logged {len(shap_paths)} SHAP figures to MLflow run {run_id}")


### Analysis — SHAP

Balance / dormancy features leading is expected given ADR 004.  
Waterfalls show *why* a single customer-month was scored high vs low — useful for Sam when reviewing retention shortlists. Paste top-5 mean |SHAP| into the model card.


## 8. Fairlearn — age & state slices + Equalized Odds (Jamie)

Disaggregated **selection rate**, **recall (TPR)**, and **accuracy** by age band and state.  
Also compute **Equalized Odds** difference/ratio (max gap in TPR/FPR across groups) to audit systemic bias. Sampled for runtime.


In [ ]:
import json
import math

from fairlearn.metrics import (
    equalized_odds_difference,
    equalized_odds_ratio,
    false_positive_rate,
)

# Ensure RNG exists even if SHAP cell was skipped.
if "rng" not in globals():
    rng = np.random.default_rng(RANDOM_STATE)

MIN_GROUP_N = 500  # board language: exclude sparse slices (ML_Learnings §2.6)

fair_n = min(FAIRNESS_SAMPLE_N, len(test_df))
fair_idx = rng.choice(len(test_df), size=fair_n, replace=False)
y_f = np.asarray(y_test)[fair_idx]
p_f = np.asarray(test_pred)[fair_idx]
age = test_df["age_years"].to_numpy()[fair_idx]
state = test_df["state"].astype(str).to_numpy()[fair_idx]

age_band = pd.cut(
    age,
    bins=[0, 25, 35, 45, 55, 65, 200],
    labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"],
    include_lowest=True,
)
# MetricFrame / equalized odds choke on NaN sensitive features.
valid = age_band.notna()
y_f, p_f, age_band, state = y_f[valid], p_f[valid], age_band[valid], state[valid]
age_band = age_band.astype(str)

metrics_fair = {
    "selection_rate": selection_rate,
    "recall": true_positive_rate,
    "fpr": false_positive_rate,
    "accuracy": accuracy_score,
}

mf_age = MetricFrame(
    metrics=metrics_fair,
    y_true=y_f,
    y_pred=p_f,
    sensitive_features=age_band,
)
mf_state = MetricFrame(
    metrics=metrics_fair,
    y_true=y_f,
    y_pred=p_f,
    sensitive_features=pd.Series(state, name="state"),
)

print("=== By age band ===")
print(mf_age.by_group)
sel_diff_age = float(mf_age.difference()["selection_rate"])
print(f"\nselection_rate difference (max-min)={sel_diff_age:.4f}")


def _safe_eod(y_true, y_pred, sf):
    try:
        return float(equalized_odds_difference(y_true, y_pred, sensitive_features=sf))
    except Exception as exc:  # noqa: BLE001
        print(f"equalized_odds_difference failed: {type(exc).__name__}: {exc}")
        return float("nan")


def _safe_eor(y_true, y_pred, sf):
    # Ratio is undefined when a group has 0 TPR or 0 FPR — Fairlearn may raise or return inf.
    try:
        val = float(equalized_odds_ratio(y_true, y_pred, sensitive_features=sf))
        if not math.isfinite(val):
            return float("nan")
        return val
    except Exception as exc:  # noqa: BLE001
        print(f"equalized_odds_ratio failed: {type(exc).__name__}: {exc}")
        return float("nan")


def _filter_min_n(y_true, y_pred, sf, min_n: int):
    """Drop sparse sensitive groups before Equalized Odds (avoids EOD=1.0 artifacts)."""
    s = pd.Series(sf)
    counts = s.value_counts()
    keep = set(counts[counts >= min_n].index.astype(str))
    mask = s.astype(str).isin(keep).to_numpy()
    meta = {
        "n_groups_total": int(counts.shape[0]),
        "n_groups_kept": len(keep),
        "n_rows_excluded": int((~mask).sum()),
        "min_group_n": min_n,
    }
    return y_true[mask], y_pred[mask], s.astype(str).to_numpy()[mask], meta


y_age, p_age, sf_age, age_filter = _filter_min_n(y_f, p_f, age_band, MIN_GROUP_N)
y_st, p_st, sf_st, state_filter = _filter_min_n(y_f, p_f, state, MIN_GROUP_N)

eod_age = _safe_eod(y_age, p_age, sf_age)
eor_age = _safe_eor(y_age, p_age, sf_age)
eod_state = _safe_eod(y_st, p_st, sf_st)
eor_state = _safe_eor(y_st, p_st, sf_st)
print(f"\n=== Equalized Odds (groups with n >= {MIN_GROUP_N}) ===")
print(f"age   difference={eod_age:.4f}  ratio={eor_age:.4f}  filter={age_filter}")
print(f"state difference={eod_state:.4f}  ratio={eor_state:.4f}  filter={state_filter}")

state_sel = mf_state.by_group["selection_rate"].sort_values(ascending=False)
print("\n=== State selection_rate (top 5 / bottom 5) ===")
print(state_sel.head(5))
print("...")
print(state_sel.tail(5))

fair_path = REPORT_DIR / "fairlearn_age_state_slices.csv"
by_age = mf_age.by_group.copy()
by_age.insert(0, "slice_type", "age_band")
age_counts = pd.Series(age_band).value_counts()
by_age["n"] = by_age.index.map(lambda g: int(age_counts.get(g, 0)))
by_age["meets_min_n"] = by_age["n"] >= MIN_GROUP_N

by_state = mf_state.by_group.copy()
by_state.insert(0, "slice_type", "state")
state_counts = pd.Series(state).value_counts()
by_state["n"] = by_state.index.map(lambda g: int(state_counts.get(g, 0)))
by_state["meets_min_n"] = by_state["n"] >= MIN_GROUP_N
pd.concat([by_age, by_state]).to_csv(fair_path)

fair_summary = {
    "n_sample": int(valid.sum()),
    "min_group_n": MIN_GROUP_N,
    "selection_rate_diff_age": sel_diff_age,
    "equalized_odds_difference_age": eod_age,
    "equalized_odds_ratio_age": eor_age,
    "equalized_odds_difference_state": eod_state,
    "equalized_odds_ratio_state": eor_state,
    "age_filter": age_filter,
    "state_filter": state_filter,
    "threshold": float(best_t) if "best_t" in globals() else None,
}
fair_summary_path = REPORT_DIR / "fairlearn_equalized_odds.json"
fair_summary_path.write_text(json.dumps(fair_summary, indent=2, allow_nan=True), encoding="utf-8")

# MLflow rejects non-finite metric values — only log finite ones.
fair_metrics = {
    "fair_selection_rate_diff_age": sel_diff_age,
    "fair_eod_age": eod_age,
    "fair_eor_age": eor_age,
    "fair_eod_state": eod_state,
    "fair_eor_state": eor_state,
    "fair_state_groups_kept": float(state_filter["n_groups_kept"]),
    "fair_state_groups_total": float(state_filter["n_groups_total"]),
}
fair_metrics = {k: v for k, v in fair_metrics.items() if isinstance(v, (int, float)) and math.isfinite(v)}

with mlflow.start_run(run_id=run_id):
    if fair_metrics:
        mlflow.log_metrics(fair_metrics)
    mlflow.log_artifact(str(fair_path), artifact_path="fairlearn")
    mlflow.log_artifact(str(fair_summary_path), artifact_path="fairlearn")

print(f"\nWrote {fair_path}")
print(f"Wrote {fair_summary_path}")


### Analysis — fairness

Equalized Odds is computed only for groups with **n ≥ 500** so sparse states cannot force EOD = 1.0.
Large remaining gaps still need a business read (true prevalence differences vs model bias).
Flag age EOD > ~0.10 for Jamie before automating outreach volume by cohort/geography.


## 9. Evidently — train vs test drift (Morgan)

Compare fit vs test feature distributions with `DataDriftPreset`.  
Interactive HTML is saved at **`Docs/Sprint2/data_drift_report.html`** (plus a copy under `reports/`) and logged to MLflow.


In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

drift_cols = [
    "age_years",
    "Months_Since_Creation",
    "Running_Balance",
    "Running_Balance_Rounded",
    "Monthly_Active_Txn_Count",
    "Monthly_Amount_Sum",
    "gone_this_month",
    "Balance_MoM_Change",
    "Active_Txn_Lag1",
    "Balance_Roll3_Mean",
    "Consecutive_Inactive_Months",
    TARGET,
]
# Clip extreme balances for the drift report only (trees already saw raw values).
ref_sample = fit_df[drift_cols].sample(n=min(50_000, len(fit_df)), random_state=RANDOM_STATE).copy()
cur_sample = test_df[drift_cols].sample(n=min(50_000, len(test_df)), random_state=RANDOM_STATE).copy()
for col in ("Running_Balance", "Running_Balance_Rounded", "Balance_Roll3_Mean", "Balance_MoM_Change"):
    ref_sample[col] = ref_sample[col].clip(-1e6, 1e6)
    cur_sample[col] = cur_sample[col].clip(-1e6, 1e6)

drift_report = Report([DataDriftPreset()])
drift_snapshot = drift_report.run(reference_data=ref_sample, current_data=cur_sample)

drift_path = REPO_ROOT / "Docs" / "Sprint2" / "data_drift_report.html"
drift_path_reports = REPORT_DIR / "evidently_train_test_drift.html"
drift_snapshot.save_html(str(drift_path))
drift_snapshot.save_html(str(drift_path_reports))

with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(drift_path), artifact_path="evidently")

print(f"Wrote {drift_path}")
print(f"Wrote {drift_path_reports}")


### Analysis — drift

Open `Docs/Sprint2/data_drift_report.html` for Morgan/Sam. If balance or activity columns drift hard into the test decade, expect some metric drop vs val — document in the model card rather than silently retuning on test.


## 10. Optional — small surrogate tree (dtreeviz)

A depth-3 surrogate on a sample helps non-technical stakeholders see a readable decision path. Fail soft if Graphviz is unavailable under AppLocker.

In [ ]:
try:
    from IPython.display import display
    from sklearn.tree import DecisionTreeClassifier
    from dtreeviz import model as dtreeviz_model

    n_surr = min(30_000, X_fit_m.shape[0])
    s_idx = rng.choice(X_fit_m.shape[0], size=n_surr, replace=False)
    sur = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE, class_weight="balanced")
    sur.fit(X_fit_m[s_idx], y_fit.to_numpy()[s_idx])
    print("Surrogate tree top features (impurity):")
    imp = pd.Series(sur.feature_importances_, index=model_feature_names).sort_values(ascending=False)
    print(imp.head(8).to_string())

    viz = dtreeviz_model(
        sur,
        X_fit_m[s_idx],
        y_fit.to_numpy()[s_idx],
        feature_names=model_feature_names,
        class_names=["stay", "churn"],
        target_name=TARGET,
    )
    display(viz.view())
except Exception as exc:  # noqa: BLE001 — optional viz must not fail the notebook
    print(f"dtreeviz skipped ({type(exc).__name__}: {exc})")
    print("Install/configure Graphviz if you need the diagram; metrics above still stand.")

## 11. Sprint 2 run summary (paste into model card)

After Run All, copy the printed block into `Docs/Model_Card_Template.md` (metrics + top SHAP features).

In [ ]:
import json

summary = {
    "run_id": run_id,
    "test_months": f"{test_months[0]} -> {test_months[-1]}",
    "threshold": best_t,
    "threshold_rule": threshold_rule if "threshold_rule" in globals() else "unknown",
    "xgboost": xgb_test,
    "baseline_heuristic": base_heur,
    "beats_heuristic_pr_auc": beat_pr,
    "beats_heuristic_roc_auc": beat_roc,
    "top_shap": shap_rank.head(5).round(6).to_dict() if "shap_rank" in globals() else {},
    "fairlearn": fair_summary if "fair_summary" in globals() else {},
    "artifacts": {
        "yellowbrick": [p.name for p in yb_paths] if "yb_paths" in globals() else [],
        "shap": [p.name for p in shap_paths] if "shap_paths" in globals() else [],
        "drift_html": "Docs/Sprint2/data_drift_report.html",
        "fairlearn_csv": "Docs/Sprint2/reports/fairlearn_age_state_slices.csv",
    },
}
print("=== MODEL CARD SNIPPET ===")
for k, v in summary.items():
    print(f"{k}: {v}")

# Ops-threshold runs write v2; do not overwrite the frozen first-run summary.
summary_path = REPORT_DIR / "sprint2_run_summary.json"
v2_path = REPORT_DIR / "sprint2_run_summary_v2.json"
rule = str(summary.get("threshold_rule", ""))
if "prec>=" in rule or "fallback" in rule or "quantile" in rule:
    v2_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    artifact = v2_path
    print(f"Wrote {v2_path}")
else:
    summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    artifact = summary_path
    print(f"Wrote {summary_path}")

with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(artifact), artifact_path="summary")

print(f"Logged {artifact.name} to MLflow")
print("Done. Next: fill Docs/Model_Card_Template.md from this snippet / ML_Learnings §2.")


## Conclusion

| Checkpoint | Expected after Run All |
|------------|------------------------|
| Temporal split | fit < val < test on `Month` |
| Baselines logged | always-negative + `gone_this_month` heuristic |
| XGBoost | PR-AUC / ROC-AUC beat heuristic |
| MLflow | SQLite `mlflow.db` experiment `meridian_churn_sprint2` + Yellowbrick/SHAP/Fairlearn/Evidently artifacts |
| Yellowbrick | ClassificationReport, ROCAUC, PrecisionRecallCurve, ConfusionMatrix |
| SHAP | summary + high/low-risk waterfalls |
| Fairlearn | age/state slices + Equalized Odds |
| Evidently | `Docs/Sprint2/data_drift_report.html` |

**Do not shuffle.** Re-run with `py -3.13` kernel (AppLocker). Heavy cells are sampled deliberately so one Run All finishes in a reasonable time.
